# 🕉️ Divya Mangalam — Gemini Vision Agent

Uses Google Gemini Managed Agents (Interactions API) to:
1. **Generate sacred devotional scripts** for daily deity videos (Sunday → Surya, Monday → Shiva, etc.).
2. **Generate Instagram, YouTube & Reels captions** with optimal hashtags.
3. **Analyze panchang timings** and produce auspicious / inauspicious time guidance.
4. **Optionally fall back to local Ollama** (llama3.2:3b) when Colab AI quota is exhausted.

**Cost: ₹0** — Uses the Free Tier of Gemini API (15 requests/min, 1M tokens/day).

Built for: `https://divyamangalam.com` | Instagram: `@divyamangalam.info`

## 🔧 Step 1: Install Dependencies

In [ ]:
# Install the latest Google GenAI SDK
%pip install -q -U "google-genai>=2.9.0" pillow requests
print('Dependencies installed!')

## 🔑 Step 2: Configure API Key (FREE Gemini API)

Get your FREE API key at: **https://aistudio.google.com/app/apikey**

Then add it to Colab Secrets:
- Click the 🔑 **Secrets** tab (left sidebar)
- Add `GEMINI_API_KEY` → paste your key
- Enable notebook access

In [ ]:
import os
import sys
import datetime
import urllib.request
import json

# ── CONFIGURATION ──────────────────────────────────────────────────────────────
WEBSITE_URL = 'https://divyamangalam.com'
INSTAGRAM_HANDLE = '@divyamangalam.info'
YOUTUBE_CHANNEL = '@DivyaMangalam'

# Try Colab Secrets first, fallback to environment variable
GEMINI_API_KEY = None
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print(f'Loaded GEMINI_API_KEY from Colab Secrets (length: {len(GEMINI_API_KEY)})')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
    if GEMINI_API_KEY:
        print(f'Loaded GEMINI_API_KEY from environment (length: {len(GEMINI_API_KEY)})')
    else:
        print('[WARNING] No GEMINI_API_KEY found. Will attempt Ollama fallback.')

# ── OLLAMA FALLBACK CONFIG ─────────────────────────────────────────────────────
OLLAMA_URL = 'http://127.0.0.1:11434/api/generate'
OLLAMA_MODEL = 'llama3.2:3b'
USE_OLLAMA = not bool(GEMINI_API_KEY)

print(f'Mode: {"LOCAL OLLAMA" if USE_OLLAMA else "GEMINI API (Free Tier)"}')

## 🧠 Step 3: Setup Gemini Client with Managed Agent

In [ ]:
import uuid

# The base Gemini managed agent (FREE quota: 15 rpm, 1M tokens/day)
BASE_AGENT = 'antigravity-preview-05-2026'
SESSION_SUFFIX = uuid.uuid4().hex[:8]
DIVYA_AGENT_ID = f'divyamangalam-vision-{SESSION_SUFFIX}'

gemini_client = None
divya_agent = None

if not USE_OLLAMA:
    from google import genai
    from google.genai import types
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)

    # Create a named Divya Mangalam Vision Agent with system instructions
    DIVYA_SYSTEM_PROMPT = """
You are the sacred content strategist for Divya Mangalam (divyamangalam.com).
You specialize in:
1. Writing devotional scripts for deity videos (Surya, Shiva, Hanuman, Ganesha, Lakshmi, etc.)
2. Crafting viral Instagram Reels and YouTube Shorts captions with Sanskrit shlokas
3. Generating panchang-based auspicious timing guidance in English
4. Creating SEO-rich YouTube video titles, descriptions, and tag lists

Style: Soulful, devotional, peaceful. No robotic language. Mix Sanskrit shlokas with plain English guidance.
Brand: Always include 'Divya Mangalam', divyamangalam.com, and @divyamangalam.info.
Target Audience: Hindu devotees in India and diaspora. Ages 25-60. Instagram + YouTube.
Hashtags: Always include #DivyaMangalam, #SanatanaDharma, and 10-15 relevant deity hashtags.
"""

    try:
        divya_agent = gemini_client.agents.create(
            id=DIVYA_AGENT_ID,
            base_agent=BASE_AGENT,
            system_instruction=DIVYA_SYSTEM_PROMPT
        )
        print(f'Divya Mangalam Vision Agent created! ID: {DIVYA_AGENT_ID}')
    except Exception as e:
        print(f'Agent creation failed: {e}')
        print('Falling back to base agent...')
        DIVYA_AGENT_ID = BASE_AGENT
else:
    print('Ollama mode: Using llama3.2:3b locally (http://127.0.0.1:11434)')

## 🌅 Step 4: Smart Query Engine (Gemini ↔ Ollama fallback)

In [ ]:
def ask_divya_agent(prompt, environment_id=None, label='Query'):
    """
    Send a prompt to Gemini Vision Agent.
    Automatically falls back to local Ollama if Gemini is unavailable.
    """
    print(f'\n[{label}] Sending to {"Gemini" if not USE_OLLAMA else "Ollama"}...')

    # ── GEMINI PATH ───────────────────────────────────────────────────────────
    if not USE_OLLAMA and gemini_client:
        try:
            kwargs = {
                'agent': DIVYA_AGENT_ID,
                'input': prompt,
                'environment': environment_id or 'remote',
            }
            interaction = gemini_client.interactions.create(**kwargs)
            print(f'  Status: {interaction.status} | ID: {interaction.id}')
            return interaction.output_text, interaction.environment_id
        except Exception as e:
            print(f'  Gemini failed ({e}), switching to Ollama...')

    # ── OLLAMA FALLBACK PATH ──────────────────────────────────────────────────
    full_prompt = (
        'You are the content strategist for Divya Mangalam (divyamangalam.com), '
        'a sacred Hindu devotional platform for Instagram and YouTube. '
        'Write with soulful, devotional, and peaceful tone. '
        'Always include @divyamangalam.info and divyamangalam.com.\n\n'
        + prompt
    )
    payload = json.dumps({
        'model': OLLAMA_MODEL,
        'prompt': full_prompt,
        'stream': False,
        'options': {'temperature': 0.7, 'num_predict': 400}
    }).encode()
    req = urllib.request.Request(OLLAMA_URL, data=payload,
                                  headers={'Content-Type': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            result = json.loads(resp.read())
            text = result.get('response', '').strip()
            print(f'  Ollama response ({len(text)} chars)')
            return text, None
    except Exception as e:
        return f'[Error: Both Gemini and Ollama unavailable. {e}]', None

print('ask_divya_agent() function ready!')

## 📅 Step 5: Today's Deity + Panchang Auto-Detection

In [ ]:
# Vedic Deity → Day of Week Mapping
DEITY_MAP = {
    6: {'name': 'Lord Surya Narayana', 'day': 'Ravivara (Sunday)', 'mantra': 'Om Suryaya Namah',
        'stotram': 'Aditya Hridaya Stotram', 'color': '#FF9900'},
    0: {'name': 'Lord Shiva (Mahadeva)', 'day': 'Somavara (Monday)', 'mantra': 'Om Namah Shivaya',
        'stotram': 'Shiva Tandava Stotram / Maha Mrityunjaya', 'color': '#0EA5E9'},
    1: {'name': 'Lord Hanuman (Bajrangbali)', 'day': 'Mangalavara (Tuesday)', 'mantra': 'Om Hanumate Namah',
        'stotram': 'Hanuman Chalisa / Sankat Mochan', 'color': '#EA580C'},
    2: {'name': 'Lord Ganesha (Ganapati)', 'day': 'Budhavara (Wednesday)', 'mantra': 'Om Gam Ganapataye Namah',
        'stotram': 'Ganesha Atharvashirsha / Vakratunda Stotram', 'color': '#7C3AED'},
    3: {'name': 'Lord Vishnu (Brihaspati)', 'day': 'Guruvara (Thursday)', 'mantra': 'Om Namo Narayanaya',
        'stotram': 'Vishnu Sahasranama / Narayana Suktam', 'color': '#0D9488'},
    4: {'name': 'Maa Lakshmi & Durga', 'day': 'Sukravara (Friday)', 'mantra': 'Om Shreem Mahalakshmyai Namah',
        'stotram': 'Mahalakshmi Ashtakam / Durga Suktam', 'color': '#DB2777'},
    5: {'name': 'Lord Shani & Hanuman', 'day': 'Shanivara (Saturday)', 'mantra': 'Om Sham Shanicharaya Namah',
        'stotram': 'Shani Chalisa / Dasaratha Shani Stotram', 'color': '#475569'},
}

today = datetime.date.today()
deity = DEITY_MAP[today.weekday()]

print(f"Today: {today.strftime('%A, %d %B %Y')}")
print(f"Vedic Day: {deity['day']}")
print(f"Presiding Deity: {deity['name']}")
print(f"Mantra: {deity['mantra']}")
print(f"Recommended Stotram: {deity['stotram']}")

## 🎬 Step 6: Generate Video Script (60-Second Devotional)

In [ ]:
SCRIPT_PROMPT = f"""
Write a 60-second devotional video script for {deity['name']} for a YouTube Short / Instagram Reel.

Today is {deity['day']} ({today.strftime('%d %B %Y')}).
Deity Mantra: {deity['mantra']}
Sacred Hymn: {deity['stotram']}

Structure the script as 4 phases of 15 seconds each:
- Phase 1 (0-15s): Opening mantra invocation + describe the divine form of {deity['name']}
- Phase 2 (15-30s): Benefits of worshipping this deity + associated sacred story / legend
- Phase 3 (30-45s): Auspicious timing guidance for today from a Vedic panchang perspective
- Phase 4 (45-60s): Closing blessing + call to action for Divya Mangalam website

Rules:
- Soulful, calming tone (NOT robotic)
- Mix Sanskrit mantras with English sentences
- Include timings: Abhijit Muhurta (midday) + Rahu Kaal caution
- End with: Visit divyamangalam.com and follow @divyamangalam.info
- Total: 100-120 words, perfect for 60 seconds of narration
"""

script_text, env_id = ask_divya_agent(SCRIPT_PROMPT, label='Video Script')
print('\n========= GENERATED SCRIPT =========')
print(script_text)

## 📸 Step 7: Generate Instagram Caption + Hashtags

In [ ]:
CAPTION_PROMPT = f"""
Write a powerful Instagram Reel caption for a devotional video about {deity['name']} ({deity['day']}).

Requirements:
- Start with a Sanskrit shloka (2 lines) related to {deity['name']}
- 3-4 paragraphs: divine intro, today's significance, auspicious timings, blessings
- Include: Abhijit Muhurta and Rahu Kaal timings for {deity['day']}
- CTA: Visit divyamangalam.com and follow @divyamangalam.info
- End with 15 high-reach hashtags including #DivyaMangalam, #SanatanaDharma
- Total caption: 300-400 characters max for 3 dots readability

Make it feel personal, warm, and spiritually uplifting — not marketing-heavy.
"""

caption_text, _ = ask_divya_agent(CAPTION_PROMPT, environment_id=env_id, label='Instagram Caption')
print('\n========= INSTAGRAM CAPTION =========')
print(caption_text)

## 🎬 Step 8: Generate YouTube Video Title + Description (SEO)

In [ ]:
YOUTUBE_PROMPT = f"""
Generate full YouTube video metadata for a Divya Mangalam devotional video about {deity['name']}.

Produce:
1. TITLE: Under 70 characters. Start with Lord's name. Include 'Divya Mangalam'. Add a power word.
2. DESCRIPTION: 300 words. Include:
   - Sacred story or context about {deity['name']}
   - Today's panchang timings (Abhijit Muhurta + Rahu Kaal)
   - Chapter timestamps (00:00 Invocation, 00:15 Significance, 00:30 Timings, 00:45 Blessings)
   - Links: https://divyamangalam.com | Instagram @divyamangalam.info
   - Subscribe CTA + #Shorts disclaimer
3. TAGS: 20 SEO-optimized tags (comma-separated)

Deity mantra: {deity['mantra']}
Sacred hymn: {deity['stotram']}
"""

youtube_meta, _ = ask_divya_agent(YOUTUBE_PROMPT, environment_id=env_id, label='YouTube Metadata')
print('\n========= YOUTUBE METADATA =========')
print(youtube_meta)

## 💾 Step 9: Export Generated Content to Files

In [ ]:
import os

day_str = today.strftime('%Y-%m-%d')
deity_slug = deity['name'].split('(')[0].strip().lower().replace(' ', '_')

output_dir = f'/content/divyamangalam_content/{day_str}'
os.makedirs(output_dir, exist_ok=True)

with open(f'{output_dir}/{deity_slug}_script.txt', 'w', encoding='utf-8') as f:
    f.write(script_text)

with open(f'{output_dir}/{deity_slug}_instagram_caption.txt', 'w', encoding='utf-8') as f:
    f.write(caption_text)

with open(f'{output_dir}/{deity_slug}_youtube_metadata.txt', 'w', encoding='utf-8') as f:
    f.write(youtube_meta)

# Combined summary JSON
summary = {
    'date': day_str,
    'deity': deity['name'],
    'mantra': deity['mantra'],
    'stotram': deity['stotram'],
    'script': script_text,
    'instagram_caption': caption_text,
    'youtube_metadata': youtube_meta
}
with open(f'{output_dir}/content_bundle.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f'All content saved to: {output_dir}')
print('Files:')
for f in os.listdir(output_dir):
    print(f'  {f}')

## 🔄 Step 10: Download Content Bundle (Optional)

In [ ]:
# Download the complete content bundle to your computer
try:
    from google.colab import files
    files.download(f'{output_dir}/content_bundle.json')
    print('Download started!')
except ImportError:
    print(f'Not in Colab. Find your files at: {output_dir}')

---
## 📊 Full System Summary

| Component | Details |
|---|---|
| **AI Engine** | Google Gemini Managed Agent (Interactions API) |
| **Model** | `antigravity-preview-05-2026` (Free Tier) |
| **Free Quota** | 15 requests/min, 1M tokens/day |
| **Fallback** | Local Ollama `llama3.2:3b` @ `127.0.0.1:11434` |
| **Content Generated** | 60s Video Script + Instagram Caption + YouTube SEO Metadata |
| **Website** | https://divyamangalam.com |
| **Instagram** | @divyamangalam.info |
| **YouTube** | @DivyaMangalam |

**Cost: ₹0 (completely free)** — Gemini free tier is sufficient for daily content generation.

To run this daily automatically, connect this Colab notebook to **Google Cloud Scheduler** or GitHub Actions.